In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader
from joblib import Parallel, delayed

In [2]:
# ─── Utilities ────────────────────────────────────────────────────────────────
lam, p, niter = 1e4, 0.01, 10
def baseline_als(y):
    L = len(y)
    D = np.diff(np.eye(L), 2)
    D = lam * D.dot(D.T)
    w = np.ones(L)
    for _ in range(niter):
        b = np.linalg.solve(np.diag(w) + D, w * y)
        w = p * (y > b) + (1 - p) * (y < b)
    return b

def preprocess_raman_single(spectrum):
    """
    Raman tower: baseline-correct → L2-normalize → abs.
    (Same as your original preprocess_single.)
    """
    b = baseline_als(spectrum)
    c = spectrum - b
    norm = np.linalg.norm(c)
    out = c / norm if norm > 0 else c
    return np.abs(out)

def preprocess_fft_single(spectrum):
    """
    FFT tower: baseline-correct → FFT magnitude (log1p) → L2-normalize.
    Must match what you used when training siamese_mixture_fft.pth.
    """
    b = baseline_als(spectrum)
    c = spectrum - b
    fft_vals = np.fft.rfft(c)
    mag = np.abs(fft_vals)
    mag = np.log1p(mag)
    norm = np.linalg.norm(mag)
    out = mag / norm if norm > 0 else mag
    return out

def floatify_cols(df):
    new = []
    for c in df.columns:
        if c in ('Label', 'Label 1', 'Label 2'):
            new.append(c)
        else:
            new.append(float(c))
    df.columns = new

In [3]:
# ─── 1) Load reference_v2 ─────────────────────────────────────────────────────
ref_df = pd.read_csv('reference_v2.csv')
floatify_cols(ref_df)
wav_cols   = [c for c in ref_df.columns if c != 'Label']
ref_specs  = ref_df[wav_cols].values       # (n_ref_samples, n_waves)
ref_labels = ref_df['Label'].values        # (n_ref_samples,)

# Unique chemical classes
classes    = sorted(np.unique(ref_labels))
C = len(classes)
class_to_i = {c:i for i,c in enumerate(classes)}

In [4]:
# ─── 2) Generate synthetic mixtures (raw domain) ──────────────────────────────
ratios = np.arange(0.05, 1.0, 0.05)
noise_level = 0.01
n_per_ratio = 10

synth_specs = []
synth_labels = []
for (i, ci), (j, cj) in combinations(enumerate(classes), 2):
    idx_i = np.where(ref_labels == ci)[0]
    idx_j = np.where(ref_labels == cj)[0]
    for r in ratios:
        for _ in range(n_per_ratio):
            spec_i = ref_specs[np.random.choice(idx_i)]
            spec_j = ref_specs[np.random.choice(idx_j)]
            mix = r * spec_i + (1-r) * spec_j
            mix += np.random.normal(scale=noise_level, size=mix.shape)
            synth_specs.append(mix)
            synth_labels.append((ci, cj))
synth_specs = np.array(synth_specs)
print("Synthetic raw spectra:", synth_specs.shape)

Synthetic raw spectra: (12540, 1024)


In [5]:
# ─── 3) Build Raman + FFT inputs for synthetic ────────────────────────────────
synth_raman = np.vstack(
    Parallel(n_jobs=-1, verbose=5)(
        delayed(preprocess_raman_single)(spec) for spec in synth_specs
    )
)
print("Synthetic Raman preproc:", synth_raman.shape)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    3.7s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    7.6s
[Parallel(n_jobs=-1)]: Done 386 tasks      | elapsed:   12.7s
[Parallel(n_jobs=-1)]: Done 584 tasks      | elapsed:   19.0s
[Parallel(n_jobs=-1)]: Done 818 tasks      | elapsed:   26.3s
[Parallel(n_jobs=-1)]: Done 1088 tasks      | elapsed:   34.7s
[Parallel(n_jobs=-1)]: Done 1394 tasks      | elapsed:   44.4s
[Parallel(n_jobs=-1)]: Done 1736 tasks      | elapsed:   55.0s
[Parallel(n_jobs=-1)]: Done 2114 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 2528 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 2978 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done 3464 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 3986 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 4544 tasks      | e

Synthetic Raman preproc: (12540, 1024)


[Parallel(n_jobs=-1)]: Done 12540 out of 12540 | elapsed:  6.5min finished


In [6]:
synth_fft = np.vstack(
    Parallel(n_jobs=-1, verbose=5)(
        delayed(preprocess_fft_single)(spec) for spec in synth_specs
    )
)
print("Synthetic FFT preproc:", synth_fft.shape)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    3.6s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    7.5s
[Parallel(n_jobs=-1)]: Done 386 tasks      | elapsed:   12.6s
[Parallel(n_jobs=-1)]: Done 584 tasks      | elapsed:   18.8s
[Parallel(n_jobs=-1)]: Done 818 tasks      | elapsed:   26.0s
[Parallel(n_jobs=-1)]: Done 1088 tasks      | elapsed:   34.5s
[Parallel(n_jobs=-1)]: Done 1394 tasks      | elapsed:   44.2s
[Parallel(n_jobs=-1)]: Done 1736 tasks      | elapsed:   55.2s
[Parallel(n_jobs=-1)]: Done 2114 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 2528 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 2978 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done 3464 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 3986 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 4544 tasks      | e

Synthetic FFT preproc: (12540, 513)


[Parallel(n_jobs=-1)]: Done 12540 out of 12540 | elapsed:  6.5min finished


In [7]:
# ─── 4) Siamese architecture (shared definition) ─────────────────────────────
class SiameseNet(nn.Module):
    def __init__(self, input_len, embed_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1,16,7,padding=3), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16,32,5,padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear((input_len//4)*32, embed_dim),
            nn.ReLU()
        )
    def forward(self,x):
        z = self.encoder(x)
        return F.normalize(z, dim=1)

# Raman Siamese (trained on Raman features)
siamese_raman = SiameseNet(input_len=ref_specs.shape[1], embed_dim=64)
siamese_raman.load_state_dict(torch.load('siamese_mixture.pth', map_location='cpu'))
siamese_raman.eval()

# FFT Siamese (trained on FFT features)
input_len_fft = synth_fft.shape[1]
siamese_fft = SiameseNet(input_len=input_len_fft, embed_dim=64)
siamese_fft.load_state_dict(torch.load('siamese_mixture_fft.pth', map_location='cpu'))
siamese_fft.eval()

SiameseNet(
  (encoder): Sequential(
    (0): Conv1d(1, 16, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=4096, out_features=64, bias=True)
    (8): ReLU()
  )
)

In [8]:
# ─── 5) Embed synthetic mixtures with BOTH towers ────────────────────────────
with torch.no_grad():
    ten_raman = torch.tensor(synth_raman, dtype=torch.float32).unsqueeze(1)
    emb_raman = siamese_raman(ten_raman).cpu().numpy()   # (N, 64)

    ten_fft   = torch.tensor(synth_fft, dtype=torch.float32).unsqueeze(1)
    emb_fft   = siamese_fft(ten_fft).cpu().numpy()       # (N, 64)

In [9]:
# Concatenate embeddings: [Emb A | Emb B]
syn_embeds = np.hstack([emb_raman, emb_fft])            # (N, 128)
print("Concatenated embeddings:", syn_embeds.shape)

Concatenated embeddings: (12540, 128)


In [11]:
# ─── 6) Build X_synth, Y_synth ────────────────────────────────────────────────
N = len(syn_embeds)
X_synth = syn_embeds
Y_synth = np.zeros((N, C), dtype=int)
for k, (ci, cj) in enumerate(synth_labels):
    Y_synth[k, class_to_i[ci]] = 1
    Y_synth[k, class_to_i[cj]] = 1

In [12]:
# ─── 7) Split into train/val/test ─────────────────────────────────────────────
X_tmp, X_test_s, Y_tmp, Y_test_s = train_test_split(
    X_synth, Y_synth, test_size=0.10, random_state=0
)
X_train_s, X_val_s, Y_train_s, Y_val_s = train_test_split(
    X_tmp, Y_tmp, test_size=0.1111, random_state=0
)
print("Synthetic train/val/test:", len(X_train_s), len(X_val_s), len(X_test_s))

Synthetic train/val/test: 10032 1254 1254


In [13]:
# ─── 8) DataLoaders ───────────────────────────────────────────────────────────
batch_size = 64
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train_s, dtype=torch.float32),
        torch.tensor(Y_train_s, dtype=torch.float32)
    ),
    batch_size=batch_size,
    shuffle=True
)
val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val_s, dtype=torch.float32),
        torch.tensor(Y_val_s, dtype=torch.float32)
    ),
    batch_size=batch_size
)
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test_s, dtype=torch.float32),
        torch.tensor(Y_test_s, dtype=torch.float32)
    ),
    batch_size=batch_size
)

In [14]:
# ─── 9) MLP on concatenated embeddings ────────────────────────────────────────
class PresenceNetLogits(nn.Module):
    def __init__(self, D, C):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, C)   # raw logits
        )
    def forward(self, x):
        return self.net(x)

D = X_train_s.shape[1]   # should be 128 (64 + 64)
C = len(classes)
model_boost = PresenceNetLogits(D, C)

In [16]:
# pos_weight (class imbalance)
pos = Y_train_s.sum(axis=0)
neg = len(Y_train_s) - pos
pos_weight = torch.tensor((neg/pos).clip(min=1.0), dtype=torch.float32)
print("pos_weight:", pos_weight)

pos_weight: tensor([4.9714, 5.0726, 5.0216, 4.9928, 5.0800, 4.9326, 5.1096, 5.0000, 4.9361,
        4.9750, 5.0144, 4.9012])


In [17]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model_boost.parameters(), lr=1e-3)

num_epochs = 200
for epoch in range(1, num_epochs+1):
    model_boost.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        logits = model_boost(xb)
        loss = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model_boost.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model_boost(xb)
            val_loss += criterion(logits, yb).item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch}/{num_epochs} — "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1/200 — Train Loss: 0.9102 | Val Loss: 0.6522
Epoch 2/200 — Train Loss: 0.5757 | Val Loss: 0.5238
Epoch 3/200 — Train Loss: 0.5025 | Val Loss: 0.4784
Epoch 4/200 — Train Loss: 0.4644 | Val Loss: 0.4454
Epoch 5/200 — Train Loss: 0.4319 | Val Loss: 0.4190
Epoch 6/200 — Train Loss: 0.4039 | Val Loss: 0.3943
Epoch 7/200 — Train Loss: 0.3809 | Val Loss: 0.3774
Epoch 8/200 — Train Loss: 0.3637 | Val Loss: 0.3589
Epoch 9/200 — Train Loss: 0.3485 | Val Loss: 0.3443
Epoch 10/200 — Train Loss: 0.3362 | Val Loss: 0.3322
Epoch 11/200 — Train Loss: 0.3264 | Val Loss: 0.3261
Epoch 12/200 — Train Loss: 0.3167 | Val Loss: 0.3150
Epoch 13/200 — Train Loss: 0.3082 | Val Loss: 0.3036
Epoch 14/200 — Train Loss: 0.3005 | Val Loss: 0.2990
Epoch 15/200 — Train Loss: 0.2925 | Val Loss: 0.2915
Epoch 16/200 — Train Loss: 0.2855 | Val Loss: 0.2835
Epoch 17/200 — Train Loss: 0.2783 | Val Loss: 0.2802
Epoch 18/200 — Train Loss: 0.2725 | Val Loss: 0.2715
Epoch 19/200 — Train Loss: 0.2667 | Val Loss: 0.2637
Ep

In [18]:
# ─── 10) Real mixtures: build Raman + FFT, embed, concat, classify ───────────
mix_df = pd.read_csv('mixtures_dataset.csv')
floatify_cols(mix_df)

wav_cols = [c for c in mix_df.columns if c not in ('Label 1', 'Label 2')]
mix_specs = mix_df[wav_cols].values.astype(float)

mix_raman = np.vstack(
    Parallel(n_jobs=-1, verbose=5)(
        delayed(preprocess_raman_single)(spec) for spec in mix_specs
    )
)
print("Mixtures Raman preproc:", mix_raman.shape)

mix_fft = np.vstack(
    Parallel(n_jobs=-1, verbose=5)(
        delayed(preprocess_fft_single)(spec) for spec in mix_specs
    )
)
print("Mixtures FFT preproc:", mix_fft.shape)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    3.6s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    7.5s
[Parallel(n_jobs=-1)]: Done 386 tasks      | elapsed:   12.5s
[Parallel(n_jobs=-1)]: Done 580 out of 580 | elapsed:   18.0s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.


Mixtures Raman preproc: (580, 1024)


[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    7.4s
[Parallel(n_jobs=-1)]: Done 386 tasks      | elapsed:   12.4s


Mixtures FFT preproc: (580, 513)


[Parallel(n_jobs=-1)]: Done 580 out of 580 | elapsed:   17.9s finished


In [19]:
with torch.no_grad():
    mix_emb_raman = siamese_raman(
        torch.tensor(mix_raman, dtype=torch.float32).unsqueeze(1)
    ).cpu().numpy()
    mix_emb_fft = siamese_fft(
        torch.tensor(mix_fft, dtype=torch.float32).unsqueeze(1)
    ).cpu().numpy()

mix_embeds = np.hstack([mix_emb_raman, mix_emb_fft])   # (N_real, 128)

pairs = list(zip(mix_df['Label 1'], mix_df['Label 2']))
N_real = len(mix_df)
Y_real = np.zeros((N_real, C), dtype=int)
for i, (l1, l2) in enumerate(pairs):
    Y_real[i, class_to_i[l1]] = 1
    Y_real[i, class_to_i[l2]] = 1

model_boost.eval()
preds = model_boost(torch.tensor(mix_embeds, dtype=torch.float32)).detach().numpy()
Y_pred_real = (preds > 0.5).astype(int)

supports = Y_real.sum(axis=0)
valid_idx = [i for i, s in enumerate(supports) if s > 0]
valid_labels = [classes[i] for i in valid_idx]

y_true_filt = Y_real[:, valid_idx]
y_pred_filt = Y_pred_real[:, valid_idx]

print("\nReal Mixtures Validation Report (labels with support > 0):")
print(classification_report(
    y_true_filt,
    y_pred_filt,
    target_names=valid_labels,
    zero_division=0
))


Real Mixtures Validation Report (labels with support > 0):
                       precision    recall  f1-score   support

      1-dodecanethiol       0.87      0.96      0.91       243
 6-mercapto-1-hexanol       0.96      0.40      0.56       108
              benzene       1.00      1.00      1.00       193
         benzenethiol       0.71      1.00      0.83        72
                 etoh       1.00      1.00      1.00       121
                 meoh       1.00      0.98      0.99       243
n,n-dimethylformamide       0.97      1.00      0.99        72
             pyridine       1.00      1.00      1.00       108

            micro avg       0.94      0.93      0.94      1160
            macro avg       0.94      0.92      0.91      1160
         weighted avg       0.95      0.93      0.93      1160
          samples avg       0.95      0.93      0.93      1160



In [20]:
# ─── 9) Evaluate synthetic test set ────────────────────────────────────────────
model_boost.eval()
yp, yt = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        yp.append(model_boost(xb).numpy())
        yt.append(yb.numpy())
y_pred = (np.vstack(yp)>0.5).astype(int)
y_true = np.vstack(yt)
print("\nSynthetic Test Classification Report:")
print(classification_report(y_true, y_pred, target_names=classes))


Synthetic Test Classification Report:
                              precision    recall  f1-score   support

           1,9-nonanedithiol       0.93      0.89      0.91       196
             1-dodecanethiol       0.55      0.95      0.69       225
             1-undecanethiol       0.51      0.69      0.59       208
        6-mercapto-1-hexanol       0.87      0.86      0.86       204
                     benzene       1.00      1.00      1.00       235
                benzenethiol       0.98      0.99      0.99       195
                        dmmp       1.00      1.00      1.00       220
                        etoh       0.95      0.99      0.97       217
                        meoh       0.97      0.94      0.95       184
       n,n-dimethylformamide       1.00      0.98      0.99       212
                    pyridine       1.00      0.99      1.00       223
tris(2-ethylhexyl) phosphate       0.99      0.96      0.98       189

                   micro avg       0.86      0.94